# Research Module 2: Data Science and AI Evaluation

**AI-Ready Radiology Curriculum**

In this notebook you will:
1. Load different types of data: CSV datasets, images, and JSON
2. Explore DataFrames and merge datasets
3. Calculate descriptive statistics (mean, median, standard deviation, IQR)
4. Create publication-quality visualizations with seaborn and matplotlib (including 3D plots)
5. Perform statistical tests (t-tests, chi-square) and interpret p-values
6. Calculate AI evaluation metrics: sensitivity, specificity, PPV, NPV, ROC/AUC
7. Conduct subgroup analysis to detect performance disparities

---

**Prerequisites:** Research Module 1 (Python basics, pandas, GitHub). If `df.head()` or `value_counts()` feel unfamiliar, revisit R1 first.

---

## Part 1: Setup and Loading Different Data Types

Research involves working with many kinds of data: spreadsheets, medical images, structured text files, and more. In this section you will load three different data types from your GitHub repository.

### 1.1 Import Libraries

We need several Python libraries. All of these come pre-installed in Google Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy import stats
import requests
from PIL import Image
from io import BytesIO
from IPython.display import display
import json

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

print('All libraries loaded successfully.')

### 1.2 Set Your GitHub Username

In [ ]:
# ============================================================
# IMPORTANT: Replace YOUR-USERNAME with your GitHub username
# ============================================================
GITHUB_USERNAME = 'YOUR-USERNAME'

BASE_URL = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/Bootcamp-AI-for-Medical-Imaging/main/data'
print(f'Base URL: {BASE_URL}')

### 1.3 Load a CSV Dataset

CSV (comma-separated values) is the most common format for tabular research data. Pandas can load a CSV directly from a URL.

In [ ]:
df = pd.read_csv(f'{BASE_URL}/radiology_ai_findings.csv')
print(f'Loaded {len(df)} studies with {df.shape[1]} columns')
df.head()

### 1.4 Load a Medical Image

Radiology research often involves images. Here we load a chest X-ray stored in your GitHub repository.

In [ ]:
sample_url = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/Bootcamp-AI-for-Medical-Imaging/main/data/CXR.jpg'

response = requests.get(sample_url)
response.raise_for_status()

image = Image.open(BytesIO(response.content))

print(f'Image size: {image.size}')
print(f'Image mode: {image.mode}')
display(image)

### 1.5 Load a Second CSV: Patient Demographics

Real research datasets often span multiple files. Here we load patient demographics that match our study IDs.

In [ ]:
demo = pd.read_csv(f'{BASE_URL}/patient_demographics.csv')
print(f'Loaded {len(demo)} patient records')
demo.head()

### 1.6 Load JSON Data

JSON (JavaScript Object Notation) is used for structured, nested data like API responses and text reports.

In [ ]:
response = requests.get(f'{BASE_URL}/ai_report_texts.json')
response.raise_for_status()
reports = response.json()

print(f'Loaded {len(reports)} AI-generated report snippets')
print(f'Data type: {type(reports)}')
print()
print('First report:')
print(json.dumps(reports[0], indent=2))

### Summary: Data Types Loaded

| Format | File | Python Type | Use Case |
|--------|------|-------------|----------|
| **CSV** | `radiology_ai_findings.csv` | pandas DataFrame | Tabular data with rows and columns |
| **Image** | `CXR.jpg` | PIL Image | Medical imaging data |
| **CSV** | `patient_demographics.csv` | pandas DataFrame | Linked metadata |
| **JSON** | `ai_report_texts.json` | list of dicts | Structured text data |

In research, you will frequently combine data from multiple sources and formats.

---

## Part 2: Exploring Your DataFrames

Before any analysis, you need to understand what your data looks like.

### 2.1 Basic Inspection

In [ ]:
print('=== DataFrame Shape ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
print()

print('=== Column Names ===')
print(list(df.columns))
print()

print('=== Data Types ===')
print(df.dtypes)
print()

print('=== First 5 Rows ===')
df.head()

In [ ]:
print('=== Last 5 Rows ===')
df.tail()

In [ ]:
print('=== DataFrame Info ===')
df.info()

### 2.2 Filtering Rows

Select specific rows using conditions.

In [ ]:
chest_only = df[df['modality'] == 'CR']
print(f'Chest X-rays: {len(chest_only)} studies')

ai_positive = df[df['ai_flagged'] == True]
print(f'AI flagged: {len(ai_positive)} studies')

chest_flagged = df[(df['modality'] == 'CR') & (df['ai_flagged'] == True)]
print(f'Chest X-rays flagged by AI: {len(chest_flagged)} studies')

### 2.3 Groupby and Aggregation

`groupby()` splits data into groups and computes statistics for each group.

In [ ]:
print('AI-flagged count per modality:')
print(df.groupby('modality')['ai_flagged'].sum())
print()

print('Mean AI confidence per modality:')
print(df.groupby('modality')['ai_confidence'].mean().round(3))
print()

print('Studies per body region:')
print(df['body_region'].value_counts())

### 2.4 Merging DataFrames

Combine the AI findings with patient demographics using a shared key (`study_id`).

In [ ]:
merged = pd.merge(df, demo, on='study_id', how='left')
print(f'Merged DataFrame: {merged.shape[0]} rows, {merged.shape[1]} columns')
merged.head()

---

## Part 3: Descriptive Statistics

Before running any tests, you should summarize your data numerically. These are the statistics that appear in the "Table 1" of every research paper.

### 3.1 Central Tendency: Mean, Median, Mode

In [ ]:
confidence = df['ai_confidence']

print('=== AI Confidence Scores ===')
print(f'Mean:   {confidence.mean():.3f}')
print(f'Median: {confidence.median():.3f}')
print(f'Mode:   {confidence.mode().values[0]:.3f}')
print()
print('Mean = average of all values')
print('Median = middle value when sorted (robust to outliers)')
print('Mode = most frequently occurring value')

### 3.2 Spread: Standard Deviation, Variance, Range, IQR

In [ ]:
print('=== Measures of Spread ===')
print(f'Standard Deviation: {confidence.std():.3f}')
print(f'Variance:           {confidence.var():.3f}')
print(f'Range:              {confidence.min():.2f} to {confidence.max():.2f} (width: {confidence.max() - confidence.min():.2f})')
print()

q25 = confidence.quantile(0.25)
q75 = confidence.quantile(0.75)
iqr = q75 - q25
print(f'25th percentile (Q1): {q25:.3f}')
print(f'75th percentile (Q3): {q75:.3f}')
print(f'IQR (Q3 - Q1):       {iqr:.3f}')
print()
print('IQR = the middle 50% of your data. Values outside Q1 - 1.5*IQR or Q3 + 1.5*IQR are often considered outliers.')

### 3.3 Summary Statistics with .describe()

In [ ]:
print('=== Numeric Summary (AI Findings) ===')
print(df.describe().round(3))
print()

print('=== Numeric Summary (Demographics) ===')
print(demo.describe().round(1))

### 3.4 Summary by Group

In [ ]:
summary = df.groupby('modality')['ai_confidence'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(3)
summary.columns = ['N', 'Mean', 'Median', 'Std Dev', 'Min', 'Max']
print('=== AI Confidence by Modality ===')
summary

---

## Part 4: Data Visualization

Visualization is how you communicate findings. We will use **seaborn** (built on matplotlib) for publication-quality plots, plus a 3D demo.

### 4.1 Histogram with KDE (Kernel Density Estimate)

Shows the distribution shape of a continuous variable.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df['ai_confidence'], bins=15, kde=True, color='#0EA5E9', edgecolor='white', ax=ax)
ax.axvline(df['ai_confidence'].mean(), color='#F43F5E', linestyle='--', linewidth=2, label=f'Mean = {df["ai_confidence"].mean():.2f}')
ax.axvline(df['ai_confidence'].median(), color='#F59E0B', linestyle='--', linewidth=2, label=f'Median = {df["ai_confidence"].median():.2f}')
ax.set_xlabel('AI Confidence Score')
ax.set_ylabel('Count')
ax.set_title('Distribution of AI Confidence Scores')
ax.legend()
plt.tight_layout()
plt.show()

### 4.2 Box Plot: Confidence by Modality

Box plots show median, quartiles, and outliers at a glance.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x='modality', y='ai_confidence', palette='Set2', ax=ax)
ax.set_xlabel('Imaging Modality')
ax.set_ylabel('AI Confidence Score')
ax.set_title('AI Confidence Distribution by Modality')
plt.tight_layout()
plt.show()

### 4.3 Violin Plot: Confidence by Body Region

Like a box plot but also shows the full distribution shape.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.violinplot(data=df, x='body_region', y='ai_confidence', palette='muted', inner='quartile', ax=ax)
ax.set_xlabel('Body Region')
ax.set_ylabel('AI Confidence Score')
ax.set_title('AI Confidence Distribution by Body Region')
plt.tight_layout()
plt.show()

### 4.4 Grouped Bar Chart

Compare counts across categories.

In [ ]:
flag_counts = df.groupby(['modality', 'ai_flagged']).size().reset_index(name='count')
flag_counts['ai_flagged'] = flag_counts['ai_flagged'].map({True: 'Flagged', False: 'Not Flagged'})

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=flag_counts, x='modality', y='count', hue='ai_flagged', palette=['#0EA5E9', '#E2E8F0'], ax=ax)
ax.set_xlabel('Imaging Modality')
ax.set_ylabel('Number of Studies')
ax.set_title('AI Flagging by Modality')
ax.legend(title='AI Decision')
plt.tight_layout()
plt.show()

### 4.5 Heatmap: Correlation Matrix

See how numeric variables relate to each other.

In [ ]:
numeric_cols = merged[['ai_confidence', 'age', 'bmi']].copy()
numeric_cols['ai_flagged'] = merged['ai_flagged'].astype(int)
numeric_cols['radiologist_confirmed'] = merged['radiologist_confirmed'].astype(int)

corr = numeric_cols.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

### 4.6 Scatter Plot with Regression Line

Explore the relationship between two continuous variables.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.regplot(data=merged, x='age', y='ai_confidence',
            scatter_kws={'alpha': 0.6, 's': 50, 'color': '#0EA5E9'},
            line_kws={'color': '#F43F5E', 'linewidth': 2}, ax=ax)
ax.set_xlabel('Patient Age')
ax.set_ylabel('AI Confidence Score')
ax.set_title('AI Confidence vs. Patient Age')
plt.tight_layout()
plt.show()

### 4.7 Count Plot: Studies by Modality and Confirmation

A seaborn count plot automatically tallies categories.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.countplot(data=df, x='modality', hue='radiologist_confirmed',
              palette={True: '#34D399', False: '#F43F5E'}, ax=ax)
ax.set_xlabel('Imaging Modality')
ax.set_ylabel('Count')
ax.set_title('Radiologist Confirmation by Modality')
ax.legend(title='Finding Confirmed', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

### 4.8 Pair Plot

A pair plot shows relationships between all pairs of numeric variables, colored by group.

In [ ]:
pair_data = merged[['ai_confidence', 'age', 'bmi', 'modality']].dropna()

g = sns.pairplot(pair_data, hue='modality', palette='Set2',
                 plot_kws={'alpha': 0.6, 's': 40},
                 height=2.5)
g.figure.suptitle('Pair Plot: Numeric Variables by Modality', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### 4.9 3D Scatter Plot

Visualize three numeric variables simultaneously. Each color represents a modality.

In [ ]:
plot_data = merged.dropna(subset=['age', 'bmi', 'ai_confidence'])
modalities = sorted(plot_data['modality'].unique())
colors = {'CR': '#0EA5E9', 'CT': '#0B1D3A', 'MR': '#F59E0B', 'US': '#34D399'}

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for mod in modalities:
    subset = plot_data[plot_data['modality'] == mod]
    ax.scatter(subset['age'], subset['bmi'], subset['ai_confidence'],
               label=mod, color=colors.get(mod, 'gray'), s=60, alpha=0.7)

ax.set_xlabel('Age')
ax.set_ylabel('BMI')
ax.set_zlabel('AI Confidence')
ax.set_title('3D: Age vs BMI vs AI Confidence')
ax.legend(title='Modality')
plt.tight_layout()
plt.show()

### 4.10 Styled Summary Table

Pandas can format DataFrames as styled tables for reports.

In [ ]:
table = merged.groupby('modality').agg(
    N=('study_id', 'count'),
    Mean_Confidence=('ai_confidence', 'mean'),
    Std_Confidence=('ai_confidence', 'std'),
    Mean_Age=('age', 'mean'),
    Pct_Flagged=('ai_flagged', 'mean')
).round(2)

table['Pct_Flagged'] = (table['Pct_Flagged'] * 100).round(1).astype(str) + '%'

table.style.set_caption('Table 1: Study Characteristics by Modality').set_table_styles(
    [{'selector': 'caption', 'props': 'font-size: 14px; font-weight: bold; margin-bottom: 8px;'}]
)

---

## Part 5: Statistical Testing

Descriptive statistics summarize data. **Statistical tests** tell you whether differences are real or could be due to chance.

### 5.1 Independent t-test: Confidence in Flagged vs. Not Flagged

A t-test compares the means of two groups and asks: "Is this difference statistically significant?"

In [ ]:
flagged = df[df['ai_flagged'] == True]['ai_confidence']
not_flagged = df[df['ai_flagged'] == False]['ai_confidence']

t_stat, p_value = stats.ttest_ind(flagged, not_flagged)

print('=== t-test: AI Confidence (Flagged vs. Not Flagged) ===')
print(f'Flagged mean:     {flagged.mean():.3f} (n={len(flagged)})')
print(f'Not flagged mean: {not_flagged.mean():.3f} (n={len(not_flagged)})')
print(f't-statistic:      {t_stat:.3f}')
print(f'p-value:          {p_value:.6f}')
print()
if p_value < 0.05:
    print('Result: Statistically significant (p < 0.05). The groups have different mean confidence scores.')
else:
    print('Result: Not statistically significant (p >= 0.05).')

### 5.2 t-test: Confidence in Confirmed vs. Not Confirmed

In [ ]:
confirmed = df[df['radiologist_confirmed'] == True]['ai_confidence']
not_confirmed = df[df['radiologist_confirmed'] == False]['ai_confidence']

t_stat2, p_value2 = stats.ttest_ind(confirmed, not_confirmed)

print('=== t-test: AI Confidence (Confirmed vs. Not Confirmed) ===')
print(f'Confirmed mean:     {confirmed.mean():.3f} (n={len(confirmed)})')
print(f'Not confirmed mean: {not_confirmed.mean():.3f} (n={len(not_confirmed)})')
print(f't-statistic:        {t_stat2:.3f}')
print(f'p-value:            {p_value2:.6f}')
print()
if p_value2 < 0.05:
    print('Result: Statistically significant (p < 0.05). Higher AI confidence is associated with real findings.')
else:
    print('Result: Not statistically significant (p >= 0.05).')

### Understanding p-values

| p-value | Interpretation |
|---------|----------------|
| p < 0.001 | Very strong evidence against null hypothesis |
| p < 0.01 | Strong evidence |
| p < 0.05 | Moderate evidence (conventional threshold) |
| p >= 0.05 | Insufficient evidence to reject null hypothesis |

The **p-value** is the probability of observing a difference this large (or larger) if there were truly no difference between the groups. It does NOT tell you the size or importance of the effect.

### 5.3 t-test: Confidence by Sex

In [ ]:
male = merged[merged['sex'] == 'M']['ai_confidence']
female = merged[merged['sex'] == 'F']['ai_confidence']

t_stat3, p_value3 = stats.ttest_ind(male, female)

print('=== t-test: AI Confidence by Patient Sex ===')
print(f'Male mean:   {male.mean():.3f} (n={len(male)})')
print(f'Female mean: {female.mean():.3f} (n={len(female)})')
print(f't-statistic: {t_stat3:.3f}')
print(f'p-value:     {p_value3:.4f}')

### 5.4 Chi-Square Test: Is AI Flagging Independent of Modality?

The chi-square test is used for categorical variables. It asks whether the distribution of one variable depends on another.

In [ ]:
contingency = pd.crosstab(df['modality'], df['ai_flagged'])
print('=== Contingency Table ===')
print(contingency)
print()

chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)

print('=== Chi-Square Test ===')
print(f'Chi-square statistic: {chi2:.3f}')
print(f'Degrees of freedom:   {dof}')
print(f'p-value:              {p_chi:.4f}')
print()
if p_chi < 0.05:
    print('Result: AI flagging rates differ significantly across modalities.')
else:
    print('Result: No significant difference in flagging rates across modalities.')

### 5.5 Visualize the t-test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(flagged, bins=12, kde=True, color='#0EA5E9', label='Flagged', alpha=0.6, ax=axes[0])
sns.histplot(not_flagged, bins=12, kde=True, color='#F43F5E', label='Not Flagged', alpha=0.6, ax=axes[0])
axes[0].set_title(f'AI Confidence: Flagged vs. Not Flagged\np = {p_value:.4f}')
axes[0].set_xlabel('AI Confidence Score')
axes[0].legend()

sns.histplot(confirmed, bins=12, kde=True, color='#34D399', label='Confirmed', alpha=0.6, ax=axes[1])
sns.histplot(not_confirmed, bins=12, kde=True, color='#F59E0B', label='Not Confirmed', alpha=0.6, ax=axes[1])
axes[1].set_title(f'AI Confidence: Confirmed vs. Not Confirmed\np = {p_value2:.4f}')
axes[1].set_xlabel('AI Confidence Score')
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.6 Summary of Statistical Tests

In [ ]:
test_results = pd.DataFrame({
    'Test': ['t-test', 't-test', 't-test', 'Chi-square'],
    'Comparison': ['Flagged vs Not Flagged', 'Confirmed vs Not Confirmed', 'Male vs Female', 'Modality vs Flagging'],
    'Statistic': [f'{t_stat:.3f}', f'{t_stat2:.3f}', f'{t_stat3:.3f}', f'{chi2:.3f}'],
    'p-value': [f'{p_value:.6f}', f'{p_value2:.6f}', f'{p_value3:.4f}', f'{p_chi:.4f}'],
    'Significant (p<0.05)': [p_value < 0.05, p_value2 < 0.05, p_value3 < 0.05, p_chi < 0.05]
})

test_results.style.set_caption('Summary of Statistical Tests').set_table_styles(
    [{'selector': 'caption', 'props': 'font-size: 14px; font-weight: bold; margin-bottom: 8px;'}]
)

---

## Part 6: AI Evaluation Metrics

Now we apply what you have learned to formally evaluate the AI system. These metrics appear in every FDA submission and published validation study.

### 6.1 Confusion Matrix

Every AI prediction falls into one of four categories:

| | Radiologist: Finding | Radiologist: Normal |
|---|---|---|
| **AI: Flagged** | True Positive (TP) | False Positive (FP) |
| **AI: Not flagged** | False Negative (FN) | True Negative (TN) |

In [ ]:
tp = len(df[(df['ai_flagged'] == True) & (df['radiologist_confirmed'] == True)])
fp = len(df[(df['ai_flagged'] == True) & (df['radiologist_confirmed'] == False)])
fn = len(df[(df['ai_flagged'] == False) & (df['radiologist_confirmed'] == True)])
tn = len(df[(df['ai_flagged'] == False) & (df['radiologist_confirmed'] == False)])

print('=== Confusion Matrix ===')
print(f'True Positives (TP):  {tp}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')
print(f'True Negatives (TN):  {tn}')
print(f'Total:                {tp + fp + fn + tn}')

In [ ]:
matrix = np.array([[tp, fp], [fn, tn]])
labels = np.array([[f'TP\n{tp}', f'FP\n{fp}'], [f'FN\n{fn}', f'TN\n{tn}']])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(matrix, annot=labels, fmt='', cmap='Blues', cbar=False,
            xticklabels=['Finding', 'Normal'],
            yticklabels=['AI: Flagged', 'AI: Not Flagged'],
            annot_kws={'size': 16, 'fontweight': 'bold'}, ax=ax)
ax.set_xlabel('Radiologist Ground Truth')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.2 Sensitivity, Specificity, PPV, NPV

| Metric | Formula | Question |
|--------|---------|----------|
| **Sensitivity** | TP / (TP + FN) | Of all real findings, how many did AI catch? |
| **Specificity** | TN / (TN + FP) | Of all normal studies, how many did AI correctly call normal? |
| **PPV** | TP / (TP + FP) | When AI flags something, how often is it real? |
| **NPV** | TN / (TN + FN) | When AI says normal, how often is it truly normal? |

In [ ]:
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)
npv = tn / (tn + fn)

print('=== Overall AI Performance ===')
print(f'Sensitivity: {sensitivity:.1%}')
print(f'Specificity: {specificity:.1%}')
print(f'PPV:         {ppv:.1%}')
print(f'NPV:         {npv:.1%}')

In [ ]:
metrics = ['Sensitivity', 'Specificity', 'PPV', 'NPV']
values = [sensitivity, specificity, ppv, npv]
bar_colors = ['#0EA5E9', '#0B1D3A', '#34D399', '#F59E0B']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(metrics, values, color=bar_colors, edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.1%}', ha='center', fontweight='bold', fontsize=12)

ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Overall AI Evaluation Metrics')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='80% threshold')
ax.legend()
plt.tight_layout()
plt.show()

### 6.3 Threshold Analysis

The AI assigns a confidence score (0.0 to 1.0). By changing the threshold for what counts as "flagged," you change the balance between sensitivity and specificity.

In [ ]:
thresholds = np.arange(0.10, 1.00, 0.05)
sens_list = []
spec_list = []

for thresh in thresholds:
    flagged_t = df['ai_confidence'] >= thresh
    t_tp = len(df[flagged_t & (df['radiologist_confirmed'] == True)])
    t_fp = len(df[flagged_t & (df['radiologist_confirmed'] == False)])
    t_fn = len(df[~flagged_t & (df['radiologist_confirmed'] == True)])
    t_tn = len(df[~flagged_t & (df['radiologist_confirmed'] == False)])
    sens = t_tp / (t_tp + t_fn) if (t_tp + t_fn) > 0 else 0
    spec = t_tn / (t_tn + t_fp) if (t_tn + t_fp) > 0 else 0
    sens_list.append(sens)
    spec_list.append(spec)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, sens_list, 'o-', color='#0EA5E9', linewidth=2, label='Sensitivity')
ax.plot(thresholds, spec_list, 's-', color='#0B1D3A', linewidth=2, label='Specificity')
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('Metric Value')
ax.set_title('Sensitivity vs. Specificity at Different Thresholds')
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.4 ROC Curve and AUC

The **ROC curve** plots sensitivity vs. (1 - specificity) at every threshold. **AUC** summarizes overall performance: 1.0 = perfect, 0.5 = random guessing.

In [ ]:
fpr_list = [1 - s for s in spec_list]

paired = sorted(zip(fpr_list, sens_list))
fpr_sorted = [p[0] for p in paired]
tpr_sorted = [p[1] for p in paired]

auc = 0
for i in range(1, len(fpr_sorted)):
    auc += (fpr_sorted[i] - fpr_sorted[i-1]) * (tpr_sorted[i] + tpr_sorted[i-1]) / 2

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(fpr_list, sens_list, 'o-', color='#0D9488', linewidth=2, markersize=4)
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, label='Random (AUC = 0.5)')
ax.fill_between(fpr_sorted, tpr_sorted, alpha=0.1, color='#0D9488')
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title(f'ROC Curve (AUC = {auc:.3f})')
ax.legend()
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'AUC = {auc:.3f}')

### 6.5 Subgroup Analysis by Modality

An AI tool might perform well overall but poorly for specific modalities. Subgroup analysis reveals these disparities.

In [ ]:
print('=== AI Performance by Modality ===')
print(f'{"Modality":<10} {"Sens":<10} {"Spec":<10} {"PPV":<10} {"NPV":<10} {"N":<5}')
print('-' * 55)

subgroup_data = []

for mod in sorted(df['modality'].unique()):
    s = df[df['modality'] == mod]
    m_tp = len(s[(s['ai_flagged']==True) & (s['radiologist_confirmed']==True)])
    m_fp = len(s[(s['ai_flagged']==True) & (s['radiologist_confirmed']==False)])
    m_fn = len(s[(s['ai_flagged']==False) & (s['radiologist_confirmed']==True)])
    m_tn = len(s[(s['ai_flagged']==False) & (s['radiologist_confirmed']==False)])
    m_sens = m_tp / (m_tp + m_fn) if (m_tp + m_fn) > 0 else 0
    m_spec = m_tn / (m_tn + m_fp) if (m_tn + m_fp) > 0 else 0
    m_ppv = m_tp / (m_tp + m_fp) if (m_tp + m_fp) > 0 else 0
    m_npv = m_tn / (m_tn + m_fn) if (m_tn + m_fn) > 0 else 0
    print(f'{mod:<10} {m_sens:<10.1%} {m_spec:<10.1%} {m_ppv:<10.1%} {m_npv:<10.1%} {len(s):<5}')
    subgroup_data.append({'Modality': mod, 'Sensitivity': m_sens, 'Specificity': m_spec})

In [ ]:
sub_df = pd.DataFrame(subgroup_data)
sub_melted = sub_df.melt(id_vars='Modality', var_name='Metric', value_name='Value')

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=sub_melted, x='Modality', y='Value', hue='Metric',
            palette=['#0EA5E9', '#0B1D3A'], ax=ax)

ax.set_ylabel('Score')
ax.set_title('AI Sensitivity and Specificity by Modality')
ax.set_ylim(0, 1.15)
ax.legend(title='Metric')

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', label_type='edge', fontsize=9, fontweight='bold',
                 padding=3)

plt.tight_layout()
plt.show()

---

## Part 7: Optional — LLM-Assisted Interpretation

Large language models (LLMs) like ChatGPT and Claude can help you draft interpretations of your results. This is an optional exercise.

### How to use an LLM for analysis

1. Copy your key results (the numbers, tables, and test outputs above)
2. Open an LLM (ChatGPT, Claude, or another)
3. Paste the template prompt below along with your results
4. **Review and verify** every number the LLM generates against your actual outputs

### Template prompt you can copy:

```
I am a medical student analyzing AI performance data from a radiology AI tool.
Here are my results:

[Paste your confusion matrix, sensitivity, specificity, PPV, NPV, AUC, and subgroup analysis outputs here]

Please help me:
1. Summarize the overall AI performance in 2-3 sentences
2. Identify which modality has the worst performance and why that matters clinically
3. Connect these findings to the concepts of never-skilling and mis-skilling
4. Suggest what threshold would be appropriate for a screening application vs. a diagnostic application
```

**Important:** LLMs can generate plausible-sounding but incorrect numbers. Always verify statistics against your notebook outputs. Use the LLM as a drafting assistant, not a calculator.

---

## Your Turn: Independent Analysis

Complete **all three tasks** below.

### Task 1: Body Region Subgroup Analysis

Pick **one body region** from the dataset. Calculate sensitivity, specificity, PPV, and NPV for that region. Create a visualization comparing it to the overall performance.

In [ ]:
# ============================================================
# TASK 1: Pick a body region and calculate all four metrics
# ============================================================
MY_REGION = 'Chest'  # <-- Change this

region_df = df[df['body_region'] == MY_REGION]
print(f'Analyzing: {MY_REGION} ({len(region_df)} studies)')

# Your code: calculate TP, FP, FN, TN for this region
# Then calculate sensitivity, specificity, PPV, NPV
# Then create a chart comparing to the overall metrics


### Task 2: Threshold Recommendation

Based on the threshold analysis in Part 6, recommend a confidence threshold for clinical use. Consider: should a screening tool prioritize sensitivity or specificity?

In [ ]:
# ============================================================
# TASK 2: Pick your recommended threshold and justify it
# ============================================================
MY_THRESHOLD = 0.70  # <-- Change this

flagged_at_thresh = df['ai_confidence'] >= MY_THRESHOLD
t_tp = len(df[flagged_at_thresh & (df['radiologist_confirmed'] == True)])
t_fp = len(df[flagged_at_thresh & (df['radiologist_confirmed'] == False)])
t_fn = len(df[~flagged_at_thresh & (df['radiologist_confirmed'] == True)])
t_tn = len(df[~flagged_at_thresh & (df['radiologist_confirmed'] == False)])

t_sens = t_tp / (t_tp + t_fn) if (t_tp + t_fn) > 0 else 0
t_spec = t_tn / (t_tn + t_fp) if (t_tn + t_fp) > 0 else 0

print(f'Recommended threshold: {MY_THRESHOLD}')
print(f'Sensitivity at {MY_THRESHOLD}: {t_sens:.1%}')
print(f'Specificity at {MY_THRESHOLD}: {t_spec:.1%}')
print(f'Studies flagged: {flagged_at_thresh.sum()} of {len(df)}')

### Task 3: Written Interpretation

In the cell below, write 3–5 sentences answering:

1. Which modality had the **highest** sensitivity? Which had the **lowest**? Why might that be?
2. What is the trade-off you observed when changing the confidence threshold?
3. How does this connect to the **never-skilling** and **mis-skilling** risks from Lesson 1?

You may optionally use an LLM (Part 7) to help draft your interpretation, but verify all numbers.

**Your interpretation:**

*Replace this text with your 3–5 sentence analysis. Reference specific numbers from your outputs above.*


---

## Save Your Work

Run the completion record cell, then save your notebook to GitHub.

### How to save:
1. In Colab, go to **File > Download > Download .ipynb**
2. Go to your forked repository: `https://github.com/YOUR-USERNAME/Bootcamp-AI-for-Medical-Imaging`
3. Click the **Code** tab at the top
4. Click **Add file > Upload files**
5. Drag and drop your downloaded `.ipynb` file into the upload area
6. In the commit message box, type: `Completed Research Module 2`
7. Make sure **"Commit directly to the main branch"** is selected
8. Click **Commit changes**

### After all research modules are complete:
When you have finished and uploaded all 5 research modules, submit your work for review:
1. On your fork's main page, click the **Contribute** button (near the top)
2. Click **Open pull request**
3. Add a title (e.g., "Completed all research modules") and click **Create pull request**

If you need to withdraw your submission, you can close the pull request at any time from the same page.

In [ ]:
from datetime import datetime

print('=' * 50)
print('RESEARCH MODULE 2 — COMPLETION RECORD')
print('=' * 50)
print(f'GitHub Username:    {GITHUB_USERNAME}')
print(f'Completed:          {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Region Analyzed:    {MY_REGION}')
print(f'Threshold Chosen:   {MY_THRESHOLD}')
print(f'Overall Sensitivity: {sensitivity:.1%}')
print(f'Overall Specificity: {specificity:.1%}')
print(f'AUC:                {auc:.3f}')
print('=' * 50)
print('Save this notebook to GitHub to submit your work.')